In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
!pip install torchinfo
import torch
from torchinfo import summary
import torch.nn as nn
from torchvision.datasets import ImageFolder
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import _LRScheduler, ReduceLROnPlateau
import torch.utils.data as data
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from tqdm import tqdm
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import random
import timm

In [ ]:
SEED = 1234

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

In [ ]:
class MultiFolderImageDataset(torch.utils.data.Dataset):
    def __init__(self, root, subfolders, transform=None):
        self.transform = transform
        self.samples = []

        for sub in subfolders:
            img_folder = os.path.join(root, sub)
            dataset = ImageFolder(img_folder)
            for path, label in dataset.samples:
                self.samples.append((path, label))

        self.classes = dataset.classes

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

In [ ]:
import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import numpy as np

batch_size = 16

# -----------------------
# 1. TRANSFORMS
# -----------------------
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=25, translate=(0.25, 0.25), scale=(0.75, 1.25)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

unnormalized_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])


# -----------------------
# 2. LOAD FULL DATASET
# -----------------------
root_dir = "/kaggle/input/breakhis-dataset-binary-multiclass-separated/dataset_cancer_v1/classificacao_binaria"   
magnifications = ["100X", "200X", "400X", "40X"]

full_dataset = MultiFolderImageDataset(root_dir, magnifications, transform=train_transform)

class_names = full_dataset.classes
print("Classes:", class_names)

N = len(full_dataset)
print("Total images =", N)


# -----------------------
# 3. SPLIT (60 / 20 / 20)
# -----------------------
train_size = int(0.6 * N)
val_size   = int(0.2 * N)
test_size  = N - train_size - val_size  # remaining 20%

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(123)
)

# Apply correct transforms
val_dataset.dataset.transform  = val_test_transform
test_dataset.dataset.transform = val_test_transform


# -----------------------
# 4. DATALOADERS
# -----------------------
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)


# -----------------------
# 5. PRINT STATS
# -----------------------
print(f"Train:      {len(train_dataset)} images")
print(f"Validation: {len(val_dataset)} images")
print(f"Test:       {len(test_dataset)} images")
print(f"Total check: {len(train_dataset)+len(val_dataset)+len(test_dataset)} images")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm


# ---------------------------
# Utility blocks
# ---------------------------

class SEVectorGate(nn.Module):
    """
    SE-style gate for vectors (B, C).
    """

    def __init__(self, dim, reduction=8):
        super().__init__()
        hidden = max(dim // reduction, 4)
        self.fc1 = nn.Linear(dim, hidden)
        self.fc2 = nn.Linear(hidden, dim)

    def forward(self, x):
        # x: (B, C)
        se = F.relu(self.fc1(x))
        se = torch.sigmoid(self.fc2(se))
        return x * se


class LinearProj(nn.Module):
    """Linear → BN → GELU projection"""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.GELU()
        )
    def forward(self, x):
        return self.net(x)


# ---------------------------
# MFB & MFH Fusion
# ---------------------------

class MFBFusion(nn.Module):
    """
    Multi-modal Factorized Bilinear Fusion (MFB)
    f_out = SumPool( (W1 f1) ⊙ (W2 f2) )  with signed-sqrt + L2 norm.
    """

    def __init__(self, in_dim, out_dim, k=5):
        super().__init__()
        self.k = k
        self.out_dim = out_dim

        self.lin1 = nn.Linear(in_dim, out_dim * k)
        self.lin2 = nn.Linear(in_dim, out_dim * k)

    def forward(self, f1, f2):
        # f1, f2: (B, D)
        a = self.lin1(f1)             # (B, out_dim * k)
        b = self.lin2(f2)             # (B, out_dim * k)

        z = a * b                     # elementwise → bilinear interaction
        z = z.view(-1, self.out_dim, self.k).sum(dim=2)  # (B, out_dim)

        # Stabilization: signed sqrt + L2 norm
        z = torch.sign(z) * torch.sqrt(torch.abs(z) + 1e-9)
        z = F.normalize(z, p=2, dim=1)

        return z                      # (B, out_dim)


class MFHFusion(nn.Module):
    """
    Multi-modal Factorized Hierarchical Fusion (MFH)
    Two-level MFB stacked → concatenated: [MFB1, MFB2].
    Output dim = 2 * out_dim.
    """

    def __init__(self, in_dim, out_dim, k=5):
        super().__init__()
        self.mfb1 = MFBFusion(in_dim, out_dim, k)
        self.mfb2 = MFBFusion(out_dim, out_dim, k)

    def forward(self, f1, f2):
        # 1st-level bilinear fusion
        m1 = self.mfb1(f1, f2)        # (B, out_dim)

        # 2nd-level hierarchical fusion (on m1 itself)
        m2 = self.mfb2(m1, m1)        # (B, out_dim)

        # Concatenate both levels
        return torch.cat([m1, m2], dim=1)   # (B, 2*out_dim)


# ---------------------------
# Dual-branch model with MFH fusion
# ---------------------------

class DualBranchEffMobNet(nn.Module):
    def __init__(self, num_classes=8,
                 proj_dim32=256,
                 proj_dim16=256,
                 proj_dim=512):
        super().__init__()

        # ---------------------------
        # Backbones
        # ---------------------------
        self.eff = timm.create_model(
            "efficientnet_b0",
            pretrained=True,
            features_only=True,
            out_indices=(2, 3, 4)     # ~32x32, 16x16, 8x8
        )
        self.mob = timm.create_model(
            "mobilenetv3_large_100",
            pretrained=True,
            features_only=True,
            out_indices=(2, 3, 4)
        )

        eff_chs = self.eff.feature_info.channels()
        mob_chs = self.mob.feature_info.channels()

        c32_e, c16_e, c8_e = eff_chs
        c32_m, c16_m, c8_m = mob_chs

        self.gap = nn.AdaptiveAvgPool2d(1)

        # -----------------------------------------
        # 32×32 fusion: GAP → proj → MFH
        # -----------------------------------------
        self.p32_e = LinearProj(c32_e, proj_dim32)
        self.p32_m = LinearProj(c32_m, proj_dim32)
        # MFH returns 2 * proj_dim32
        self.fuse32 = MFHFusion(proj_dim32, proj_dim32, k=5)

        # -----------------------------------------
        # 16×16 fusion: GAP → proj → MFH
        # -----------------------------------------
        self.p16_e = LinearProj(c16_e, proj_dim16)
        self.p16_m = LinearProj(c16_m, proj_dim16)
        self.fuse16 = MFHFusion(proj_dim16, proj_dim16, k=5)

        # -----------------------------------------
        # High-level 8×8 fusion: GAP → proj → MFH
        # -----------------------------------------
        self.proj_f1 = LinearProj(c8_e, proj_dim)
        self.proj_f2 = LinearProj(c8_m, proj_dim)
        self.fuse_high = MFHFusion(proj_dim, proj_dim, k=5)

        # MFH-high output dim = 2 * proj_dim
        self.se_vec = SEVectorGate(2 * proj_dim)

        # -----------------------------------------
        # Final aggregation
        # -----------------------------------------
        fused32_dim = 2 * proj_dim32
        fused16_dim = 2 * proj_dim16
        fused_high_dim = 2 * proj_dim

        total_dim = fused32_dim + fused16_dim + fused_high_dim
        self.se_final = SEVectorGate(total_dim)

        self.head = nn.Sequential(
            nn.Linear(total_dim, 512),
            nn.GELU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.GELU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.25),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # x: (B, 3, H, W), e.g. (B, 3, 256, 256)

        eff_feats = self.eff(x)   # [f32_e, f16_e, f8_e]
        mob_feats = self.mob(x)   # [f32_m, f16_m, f8_m]

        fe32_e, fe16_e, f1_map = eff_feats
        fe32_m, fe16_m, f2_map = mob_feats

        # ---------------------------
        # 32×32
        # ---------------------------
        f32_e = self.p32_e(self.gap(fe32_e).flatten(1))
        f32_m = self.p32_m(self.gap(fe32_m).flatten(1))
        fused32 = self.fuse32(f32_e, f32_m)       # (B, 2*proj_dim32)

        # ---------------------------
        # 16×16
        # ---------------------------
        f16_e = self.p16_e(self.gap(fe16_e).flatten(1))
        f16_m = self.p16_m(self.gap(fe16_m).flatten(1))
        fused16 = self.fuse16(f16_e, f16_m)       # (B, 2*proj_dim16)

        # ---------------------------
        # High-level 8×8
        # ---------------------------
        f1 = self.proj_f1(self.gap(f1_map).flatten(1))
        f2 = self.proj_f2(self.gap(f2_map).flatten(1))
        fused_high = self.fuse_high(f1, f2)       # (B, 2*proj_dim)
        fused_high = self.se_vec(fused_high)

        # ---------------------------
        # Final concatenation
        # ---------------------------
        fused = torch.cat([fused32, fused16, fused_high], dim=1)
        fused = self.se_final(fused)

        logits = self.head(fused)                 # (B, num_classes)
        return logits


In [ ]:
num_classes = len(class_names)  
model = DualBranchEffMobNet(num_classes=num_classes, proj_dim=512)

In [ ]:
#define LR-optimizer

FOUND_LR = 1e-3
optimizer = optim.Adam(model.parameters(), lr=FOUND_LR, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.1,      
    patience=10,       
    verbose=True
)

#Metric Computation 
def compute_metrics(y_true, y_pred, average='macro'):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average=average, zero_division=0)
    rec = recall_score(y_true, y_pred, average=average, zero_division=0)
    f1 = f1_score(y_true, y_pred, average=average, zero_division=0)
    return acc, prec, rec, f1

In [ ]:
#Train Function
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    all_preds, all_labels = [], []
    running_loss = 0.0

    for images, labels in tqdm(loader, desc="Training", leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    acc, prec, rec, f1 = compute_metrics(all_labels, all_preds)
    return epoch_loss, acc, prec, rec, f1


#Validation

def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    all_preds, all_labels = [], []
    running_loss = 0.0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validation", leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    acc, prec, rec, f1 = compute_metrics(all_labels, all_preds)
    return epoch_loss, acc, prec, rec, f1

In [ ]:
#Test Step (with Confusion Matrix)
def test_model(model, loader, criterion, device, class_names=None):
    model.eval()
    all_preds, all_labels = [], []
    running_loss = 0.0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Testing", leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = running_loss / len(loader.dataset)
    acc, prec, rec, f1 = compute_metrics(all_labels, all_preds)

    # 📊 Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(cmap=plt.cm.Blues, xticks_rotation=45)
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.show()

    return avg_loss, acc, prec, rec, f1,all_labels, all_preds

# Full Training Loop
def run_training(model, train_loader, val_loader, test_loader, optimizer, scheduler, criterion, device, class_names=None, num_epochs=200, patience=10):
    best_val_loss = float('inf')
    epochs_without_improvement = 0

    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []


    for epoch in range(num_epochs):
        train_loss, train_acc, train_prec, train_rec, train_f1 = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc, val_prec, val_rec, val_f1 = validate_one_epoch(model, val_loader, criterion, device)

        # Store for plotting
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accuracies.append(train_acc)
        val_accuracies.append(val_acc)
        
        current_lrs = [param_group['lr'] for param_group in optimizer.param_groups]
        print(f"\nEpoch [{epoch+1}/{num_epochs}] - Current LRs: {current_lrs}")
        print(f"Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, Prec: {train_prec:.4f}, Recall: {train_rec:.4f}, F1: {train_f1:.4f}")
        print(f"Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, Prec: {val_prec:.4f}, Recall: {val_rec:.4f}, F1: {val_f1:.4f}")

        scheduler.step(val_loss)
        
        # Save best model by validation loss
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_without_improvement = 0
            torch.save(model.state_dict(), 'best_attention_model.pth')
            print(f"✅ Saved best model (val_loss: {val_loss:.4f})")
        else:
            epochs_without_improvement += 1
            print(f"⏳ No improvement in val_loss for {epochs_without_improvement} epoch(s)")

            if epochs_without_improvement >= patience:
                print(f"🛑 Early stopping triggered after {patience} epochs without improvement.")
                break


    model.load_state_dict(torch.load('best_attention_model.pth'))
    print("📥 Loaded the best saved model for final testing.")

    # Final evaluation
    print("\n🔍 Evaluating on Test Set:")
    test_loss, test_acc, test_prec, test_rec, test_f1, all_labels, all_preds = test_model(model, test_loader, criterion, device, class_names)
    print(f"Test - Loss: {test_loss:.4f}, Acc: {test_acc:.4f}, Prec: {test_prec:.4f}, Recall: {test_rec:.4f}, F1: {test_f1:.4f}")

    corrects = torch.eq(torch.tensor(all_labels), torch.tensor(all_preds))
    num_correct = corrects.sum().item()
    total = len(corrects)
    print(f"✅ Correct Predictions: {num_correct}/{total} ({100 * num_correct / total:.2f}%)")

    return train_losses, val_losses, train_accuracies, val_accuracies

# Run the training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f"Using device: {device}")

# Print model summary
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total trainable parameters: {count_parameters(model):,}")

# Start training
train_losses, val_losses, train_accuracies, val_accuracies=run_training(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=torch.nn.CrossEntropyLoss(),
    device=device,
    class_names=class_names,
    num_epochs=120,
    patience=120
)
# 📈 Plot Loss Curve
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.title('Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# 📈 Plot Accuracy Curve
plt.figure(figsize=(10, 4))
plt.plot(train_accuracies, label='Train Acc')
plt.plot(val_accuracies, label='Val Acc')
plt.title('Accuracy Curve')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Save final model weights after testing
torch.save(model.state_dict(), '/kaggle/working/EfficientNet_plus_MobilenetV3Large_combined_binary_BreakHis.pth')
print("✅ Saved final model weights after testing")